In [ ]:
diff --git a/c:\Users\LENOVO\Documents\GitHub\ihsg-analysis/ihsg_analysis.py b/c:\Users\LENOVO\Documents\GitHub\ihsg-analysis/ihsg_analysis.py
new file mode 100644
--- /dev/null
+++ b/c:\Users\LENOVO\Documents\GitHub\ihsg-analysis/ihsg_analysis.py
@@ -0,0 +1,82 @@
+import sys
+from datetime import datetime, timezone
+
+import pandas as pd
+import yfinance as yf
+
+
+def fetch_ihsg(period: str = "6mo") -> pd.DataFrame:
+    # Yahoo Finance ticker for IHSG is ^JKSE.
+    df = yf.download("^JKSE", period=period, interval="1d", auto_adjust=True, progress=False)
+    if df.empty:
+        raise RuntimeError("No data returned for ^JKSE. Check network or ticker.")
+    df = df.reset_index()
+    return df
+
+
+def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
+    out = df.copy()
+    out["ma20"] = out["Close"].rolling(20).mean()
+    out["ma50"] = out["Close"].rolling(50).mean()
+    out["rsi14"] = compute_rsi(out["Close"], period=14)
+    out["ret_1d"] = out["Close"].pct_change() * 100.0
+    out["ret_5d"] = out["Close"].pct_change(5) * 100.0
+    return out
+
+
+def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
+    delta = series.diff()
+    gain = delta.clip(lower=0)
+    loss = -delta.clip(upper=0)
+    avg_gain = gain.rolling(period).mean()
+    avg_loss = loss.rolling(period).mean()
+    rs = avg_gain / avg_loss
+    rsi = 100 - (100 / (1 + rs))
+    return rsi
+
+
+def summarize(df: pd.DataFrame) -> str:
+    latest = df.iloc[-1]
+    prev = df.iloc[-2] if len(df) >= 2 else None
+
+    lines = []
+    lines.append(f"Tanggal terakhir data: {latest['Date'].date()}")
+    lines.append(f"Close terakhir: {latest['Close']:.2f}")
+
+    if prev is not None:
+        change = latest["Close"] - prev["Close"]
+        pct = (change / prev["Close"]) * 100
+        lines.append(f"Perubahan harian: {change:.2f} ({pct:.2f}%)")
+
+    lines.append(f"MA20: {latest['ma20']:.2f} | MA50: {latest['ma50']:.2f}")
+    lines.append(f"RSI14: {latest['rsi14']:.2f}")
+    lines.append(f"Return 5 hari: {latest['ret_5d']:.2f}%")
+
+    # Simple state signal for quick read
+    state = []
+    if latest["Close"] > latest["ma20"]:
+        state.append("di atas MA20")
+    else:
+        state.append("di bawah MA20")
+    if latest["Close"] > latest["ma50"]:
+        state.append("di atas MA50")
+    else:
+        state.append("di bawah MA50")
+    lines.append("Status tren: " + ", ".join(state))
+
+    return "\n".join(lines)
+
+
+def main() -> int:
+    period = sys.argv[1] if len(sys.argv) > 1 else "6mo"
+    df = fetch_ihsg(period=period)
+    df = add_indicators(df)
+
+    print("Analisis IHSG (berbasis data penutupan terakhir)")
+    print(f"Waktu eksekusi UTC: {datetime.now(timezone.utc).isoformat(timespec='seconds')}")
+    print(summarize(df))
+    return 0
+
+
+if __name__ == "__main__":
+    raise SystemExit(main())
